<!--
SPDX-License-Identifier: Apache-2.0
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
-->

# Nemotron 3.5 Lightning on NVIDIA DGX Spark

### A private, OpenAI-compatible endpoint on the box on your desk

This notebook brings up an open-weights 30B model on a single DGX Spark and drives it
from the standard OpenAI Python SDK. **Nothing leaves the machine** — no API key, no
egress, no rate limits.

---

**Everything here is open:**

| Component | License |
|---|---|
| [vLLM](https://github.com/vllm-project/vllm) | Apache 2.0 |
| [Nemotron 3.5 Lightning 30B-A3B NVFP4](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4) | OpenMDW-1.1 — weights **and training data** released |
| This sample | Apache 2.0 |

**To run it yourself on your own Spark:**

```bash
git clone https://github.com/NVIDIA/nvidia-oci-samples.git
cd nvidia-oci-samples/dgx-spark-samples/nemotron-lightning-vllm-endpoint
./setup.sh      # once
./serve.sh      # leave running
jupyter lab demo.ipynb
```

---
## 1. What is this machine?

Before anything else — confirm what we're running on.

In [ ]:
import subprocess, platform, shutil, textwrap

print("=" * 66)
print("  HOST".ljust(66))
print("=" * 66)
print(f"  hostname      {platform.node()}")
print(f"  architecture  {platform.machine()}")
print(f"  python        {platform.python_version()}")

if shutil.which("nvidia-smi"):
    q = "name,memory.total,driver_version,compute_cap"
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    for line in out.splitlines():
        name, mem, drv, cc = [p.strip() for p in line.split(",")]
        print(f"  gpu           {name}")
        print(f"  memory        {mem}")
        print(f"  driver        {drv}")
        print(f"  compute cap   {cc}   (GB10 = sm_121)")
else:
    print("  gpu           nvidia-smi not found")

print("=" * 66)
print(textwrap.dedent("""
  128 GB of unified CPU+GPU memory at 273 GB/s.
  Memory-rich, bandwidth-bound. That shapes everything below.
"""))

---
## 2. Is the endpoint up?

`serve.sh` should already be running in another terminal. Cold start is ~5 minutes,
so start it *before* you need it.

In [ ]:
import requests

BASE_URL = "http://localhost:8000/v1"
MODEL_ID = None

try:
    r = requests.get(f"{BASE_URL}/models", timeout=5)
    r.raise_for_status()
    models = r.json()["data"]
    MODEL_ID = models[0]["id"]
    print("  ENDPOINT IS UP\n")
    print(f"  url     {BASE_URL}")
    for m in models:
        print(f"  model   {m['id']}")
    print(f"  context {models[0].get('max_model_len', 'n/a'):,} tokens"
          if isinstance(models[0].get("max_model_len"), int) else "")
except Exception as e:
    print("  ENDPOINT IS NOT REACHABLE\n")
    print(f"  {type(e).__name__}: {e}\n")
    print("  Start it in another terminal:\n")
    print("      ./serve.sh\n")
    print("  Cold start takes about 5 minutes. Watch for:")
    print('      "Application startup complete"')

---
## 3. The only line that changes

Here is the whole adoption story for anyone already using a hosted model.

```python
client = OpenAI(
    base_url="https://api.openai.com/v1",     # before
    api_key=os.environ["OPENAI_API_KEY"],
)
```

```python
client = OpenAI(
    base_url="http://localhost:8000/v1",      # after — your Spark
    api_key="not-needed",
)
```

Same SDK, same request shape, same response shape. Every tool built on the OpenAI
API — LangChain, LlamaIndex, Continue, your own services — points here with a
configuration change. **That is the port.**

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=BASE_URL,        # <-- the only line that changes
    api_key="not-needed",     #     no key: it is your hardware
)

resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user",
               "content": "In two sentences: why would a company run a model "
                          "on hardware they own instead of a hosted API?"}],
    max_tokens=200,
    temperature=0.3,
)

print(resp.choices[0].message.content)
print()
print("-" * 66)
print(f"  served by     {resp.model}")
print(f"  tokens        {resp.usage.prompt_tokens} in / "
      f"{resp.usage.completion_tokens} out")

---
## 4. Streaming, with the numbers visible

Time-to-first-token and decode rate, measured live. Real hardware has jitter —
you can see it here.

In [ ]:
import time, sys

def ask(prompt, system=None, max_tokens=300, temperature=0.3, show_stats=True):
    """Stream an answer from the Spark and report TTFT + decode rate."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    t0 = time.perf_counter()
    ttft = None
    n_tokens = 0
    chunks = []

    stream = client.chat.completions.create(
        model=MODEL_ID, messages=messages,
        max_tokens=max_tokens, temperature=temperature, stream=True,
    )

    for chunk in stream:
        delta = chunk.choices[0].delta
        piece = getattr(delta, "content", None)
        if piece:
            if ttft is None:
                ttft = time.perf_counter() - t0
            n_tokens += 1
            chunks.append(piece)
            sys.stdout.write(piece)
            sys.stdout.flush()

    total = time.perf_counter() - t0
    if show_stats:
        decode = (n_tokens - 1) / (total - ttft) if ttft and total > ttft else 0.0
        print("\n" + "-" * 66)
        print(f"  TTFT {ttft*1000:>7.1f} ms   "
              f"decode {decode:>5.1f} tok/s   "
              f"total {total:>5.2f} s   "
              f"({n_tokens} tokens)")
    return "".join(chunks)


_ = ask("Explain what 'unified memory' means on a Grace Blackwell system, "
        "and why it matters for running large models. Keep it under 120 words.")

---
## 5. Your turn

Paste a prompt below and run the cell. This is going to a 30B model on one
desk-side machine, and nothing about it touches the internet.

In [ ]:
# ---------------------------------------------------------------------------
# Replace the text below and re-run.
# ---------------------------------------------------------------------------

PROMPT = "What are three things you would use a local 30B model for that you would not send to a hosted API?"

_ = ask(PROMPT, max_tokens=350)

---
## 6. It is an agent, not a chatbot

The model is served with structured tool calling enabled:

```
--enable-auto-tool-choice --tool-call-parser qwen3_coder
```

so tool calls come back as structured `tool_calls`, not text to scrape.

Below we give it two unrelated tools and a task that needs both. Nobody tells it
to chain them — it works out that it has to search first, then convert.

The tool implementations are plain local Python (`demo_tools.py`). Swap them for
Fusion APIs, an internal service, or a database and the loop is unchanged.

In [ ]:
import json
from demo_tools import TOOL_SCHEMAS, TOOL_IMPLEMENTATIONS

for t in TOOL_SCHEMAS:
    fn = t["function"]
    params = ", ".join(fn["parameters"]["properties"])
    print(f"  {fn['name']}({params})")
    print(f"      {fn['description'][:80]}")
print(f"\n  {len(TOOL_SCHEMAS)} tools available. Real functions, executed locally.")

In [ ]:
def run_agent(task, max_rounds=6, verbose=True):
    """Full tool-calling loop: model decides, we execute, model continues."""
    messages = [{"role": "user", "content": task}]
    calls_made = []

    for round_no in range(1, max_rounds + 1):
        resp = client.chat.completions.create(
            model=MODEL_ID, messages=messages,
            tools=TOOL_SCHEMAS, tool_choice="auto",
            max_tokens=1024, temperature=0.0,
        )
        msg = resp.choices[0].message

        # vLLM >= 0.26 exposes the reasoning trace as `reasoning`,
        # older builds used `reasoning_content`. Handle both.
        reasoning = getattr(msg, "reasoning", None) or \
                    getattr(msg, "reasoning_content", None)
        if reasoning and verbose:
            snippet = reasoning.strip().replace("\n", " ")[:150]
            print(f"  [round {round_no}] thinking: {snippet}...")

        if not msg.tool_calls:
            if verbose:
                print(f"\n  [round {round_no}] final answer\n")
            print("  " + "\n  ".join(msg.content.strip().split("\n")))
            return {"answer": msg.content, "calls": calls_made, "rounds": round_no}

        messages.append(msg)
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            calls_made.append(name)
            if verbose:
                print(f"  [round {round_no}] CALL  {name}({json.dumps(args)})")

            impl = TOOL_IMPLEMENTATIONS.get(name)
            result = impl(**args) if impl else {"error": f"unknown tool {name}"}

            if verbose:
                print(f"  [round {round_no}] ->    {json.dumps(result)[:110]}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })

    return {"answer": None, "calls": calls_made, "rounds": max_rounds}


TASK = ("Find the cheapest flight from SFO to AUS on 2026-09-14, "
        "then tell me that fare in euros.")

print(f"  TASK: {TASK}\n")
print("=" * 66)
result = run_agent(TASK)
print("=" * 66)
print(f"\n  {len(result['calls'])} tool calls across {result['rounds']} rounds: "
      f"{' -> '.join(result['calls'])}")

---
## 7. The harder question: does it know when *not* to call?

The agent loop above is an anecdote. This is the evidence.

Most function-calling benchmarks score whether a call was *correct* once the model
decided to make one. [`nvidia/When2Call`](https://huggingface.co/datasets/nvidia/When2Call)
(CC-BY-4.0) scores something more useful: **whether it should have called anything at all.**

| Label | Correct behaviour |
|---|---|
| `tool_call` | Call the tool — every required argument is available |
| `request_for_info` | Ask a follow-up — a required argument was deliberately withheld |
| `cannot_answer` | Decline — no available tool covers the request |

Scoring is deterministic — we observe what the server did, no LLM judge — so you can
reproduce these numbers on your own hardware. Four metrics:

- **Decision accuracy** — did it call exactly when it should have?
- **Actionable decision accuracy** — correct decision *and* it either called a tool or
  produced real text. Stops a model scoring well by staying silent.
- **Tool-selection accuracy** — when it correctly called, was it the right tool?
- **Over-call rate** — how often it fired a tool it shouldn't have. **Lower is better.
  This is the number that predicts agent misbehaviour in production.**

### 7a. Five models, same 120 examples

Loaded from `results/summary.json` — generated by `./run_benchmark.py`, which you can
run yourself. Identical prompts and identical converted tool schemas for every model.

In [ ]:
import json, pathlib
from when2call import pct, overlaps

summary = json.loads(pathlib.Path("results/summary.json").read_text())
models  = sorted(summary["models"],
                 key=lambda m: -m["decision_accuracy"]["rate"])

print(f"  nvidia/When2Call · {summary['n_examples']} examples · run {summary['generated']}\n")
print(f"  {'model':<20}{'decision acc':<26}{'tool sel':<12}{'over-call'}")
print("  " + "-" * 72)
for m in models:
    d, t, o = (m["decision_accuracy"], m["tool_selection_accuracy"],
               m["over_call_rate"])
    star = " *" if "local" in m.get("location", "").lower() else "  "
    print(f"  {m['name']:<18}{star}"
          f"{d['k']:>3}/{d['n']:<3} {pct(d['rate']):>6} "
          f"[{pct(d['lo'])}–{pct(d['hi'])}]".ljust(26)
          + f"{t['k']:>2}/{t['n']:<3}      "
          + f"{pct(o['rate']):>6}")

for f in summary.get("failed", []):
    print(f"  {f['name']:<20}  no data — {f['reason'][:44]}")

print("\n  * running locally on this DGX Spark")

In [ ]:
from IPython.display import Image, display
display(Image(filename="results/when2call.png"))

### 7b. Reading it honestly

Two things worth saying out loud, because the chart invites a wrong reading.

In [ ]:
top = models[0]["decision_accuracy"]
tied = [m["name"] for m in models
        if overlaps(m["decision_accuracy"], top)]

print("  1. DECISION ACCURACY IS A TIE.")
print(f"     {', '.join(tied)}")
print("     have overlapping 95% confidence intervals. At n=120 that is roughly")
print("     +/-6.5 points — not enough to separate models within ~10 points of")
print("     each other. Ranking them would be overreading the data.\n")

worst = max(models, key=lambda m: m["over_call_rate"]["rate"])
best  = min(models, key=lambda m: m["over_call_rate"]["rate"])
print("  2. OVER-CALL RATE IS WHERE A REAL DIFFERENCE APPEARS.")
print(f"     {worst['name']} fires a tool it shouldn't {pct(worst['over_call_rate']['rate'])}"
      f" of the time.")
print(f"     {best['name']} does it {pct(best['over_call_rate']['rate'])} of the time,"
      f" on identical examples.")
print("     If your agent has write access, that gap is the one that decides")
print("     whether it makes bad writes.\n")

spark = next((m for m in models if "local" in m.get("location","").lower()), None)
host  = next((m for m in models if m["name"] == "nemotron-hosted"), None)
if spark and host:
    same = overlaps(spark["decision_accuracy"], host["decision_accuracy"])
    print("  CONTROL: the same model, two places.")
    print(f"     local {pct(spark['decision_accuracy']['rate'])}  vs  "
          f"hosted {pct(host['decision_accuracy']['rate'])}  ->  "
          f"{'equivalent within noise' if same else 'DIFFERENT — investigate'}")
    print("     If these two disagreed, the cross-model comparison would mean nothing.")

### 7c. Run a slice live, right now

The table above took about nine minutes for the full 120 examples. Here's a small
stratified slice against the Spark so you can watch the harness actually work —
same code, same scoring, just fewer examples.

In [ ]:
import time
import when2call as w2c
from run_benchmark import call_with_backoff

N_PER_LABEL = 4          # 12 examples — about a minute. Raise for a longer run.

examples = w2c.load_examples(n_per_label=N_PER_LABEL, seed=7)
print(f"  {len(examples)} examples, stratified across the three labels\n")

records = []
t0 = time.perf_counter()
for i, ex in enumerate(examples, 1):
    msg, finish, err = call_with_backoff(
        client, model=MODEL_ID,
        messages=w2c.build_messages(ex),
        tools=w2c.to_openai_tools(ex["tools"]),
        max_tokens=3072)
    if err:
        rec = w2c.Record(label=ex["label"], called=False, tool_name=None,
                         gold_tool=ex.get("gold_tool"), has_text=False,
                         finish_reason=None, error=err)
    else:
        rec = w2c.observe(msg, finish)
        rec.label = ex["label"]
        rec.gold_tool = ex.get("gold_tool")
    records.append(rec)

    mark = "OK  " if rec.decision_correct else "MISS"
    did  = f"called {rec.tool_name}" if rec.called else "no call"
    print(f"  [{i:>2}/{len(examples)}] {mark}  want {rec.label:<17} got {did}")

print(f"\n  {time.perf_counter()-t0:.1f}s on this box\n")
print(w2c.summarise("live slice (DGX Spark)", w2c.score(records)))
print("  Small n — wide intervals. This proves the harness runs, not a ranking.")

---
## 8. What is left over

The model is resident. How much of the box is still free?

In [ ]:
import subprocess

out = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
).stdout.strip()

if out:
    used, total = [int(x) for x in out.split(",")]
    free = total - used
    bar_w = 44
    filled = int(bar_w * used / total)
    print("  MEMORY")
    print("  [" + "#" * filled + "." * (bar_w - filled) + "]")
    print(f"  {used/1024:.1f} GB used   {free/1024:.1f} GB free   "
          f"{total/1024:.1f} GB total")

print("""
  Weights resident        17.86 GiB
  KV cache available      84.78 GiB   (~23.4M tokens)
  Cold start              ~5 minutes
  TTFT, short prompt      67 ms
  TTFT, ~8k prompt        87 ms
  Decode, batch 1         76 tok/s

  Throughput is flat from a short prompt to an 8k prompt: prefill is cheap
  relative to bandwidth-bound decode on GB10.

  30B total parameters, 3B active per token (MoE). That is why a 30B model
  serves comfortably from a desktop.
""")

---
## 9. Where this goes next

Same box, same 128 GB — inference is the easy half.

| | |
|---|---|
| **Optimize** | [Quantize to NVFP4 with Model Optimizer](https://build.nvidia.com/spark/nvfp4-quantization) — ~3.5× memory reduction vs FP16, accuracy close to FP8 |
| **Train** | [Fine-tune with PyTorch](https://build.nvidia.com/spark/pytorch-fine-tune) — FSDP + LoRA, up to 70B across two Sparks · [Unsloth](https://build.nvidia.com/spark/unsloth) · [LLaMA-Factory](https://build.nvidia.com/spark/llama-factory) |
| **Scale** | [Connect two Sparks](https://build.nvidia.com/spark/connect-two-sparks) over 200 Gb/s for 256 GB pooled |

Fine-tuning is memory-bound. Full fine-tuning of Llama 3.2 3B, LoRA on Llama 3.1 8B,
and QLoRA on Llama 3.3 70B all run here — **and none of them fit on a 32 GB consumer GPU.**
That is the argument for this box, and it is the part you cannot do against a hosted API at all.

---

### Being straight about the limits

- **A single GB10 is a development endpoint, not a shared production one.** 273 GB/s of
  memory bandwidth means concurrent requests contend and largely serialise. One user or
  one CI job: excellent. Thirty simultaneous users: not what this is for.
- **GB10 has no native FP4 compute.** NVFP4 here is a *memory* optimisation — weights
  stored 4-bit, decompressed to compute via the Marlin kernel. A large win on a
  bandwidth-bound part, but it is not FP4 tensor-core acceleration.
- **Do not buy this for throughput.** Buy it for capacity, data residency, zero marginal
  cost per token, and no rate limits.

---

### Run it yourself

```bash
git clone https://github.com/NVIDIA/nvidia-oci-samples.git
cd nvidia-oci-samples/dgx-spark-samples/nemotron-lightning-vllm-endpoint
./setup.sh && ./serve.sh
```

Questions, issues, or a sample of your own to contribute:
[github.com/NVIDIA/nvidia-oci-samples/issues](https://github.com/NVIDIA/nvidia-oci-samples/issues)